# 04 · My own games

Running my recent ranked games through the model: what did the draft say *before* the game
started, and what actually happened?

**The honesty check that makes this valid:** my games are cached into the same `data/raw/`
as the training data, so `parse.py` explicitly holds out every match id listed in
`data/processed/my_match_ids.txt`. The model is therefore structurally incapable of having
trained on the games it is judged against here.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config, features as F, viz
viz.apply_style()
pd.set_option("display.width", 140)

In [ ]:
import joblib
bundle = joblib.load(config.DATA_PROCESSED / "model.joblib")
model, columns, tier = bundle["model"], bundle["columns"], bundle["tier"]

train_ids = set(F.load_matches()["matchId"])
mine = set((config.DATA_PROCESSED / "my_match_ids.txt").read_text().split())
print(f"my cached games: {len(mine)}")
print(f"overlap with training data: {len(mine & train_ids)}  <- must be 0")

Run `python src/my_games.py --count 40` first to refresh the cache and figures. The cell below
rebuilds the per-game table using the same pipeline.

In [ ]:
import parse, my_games as MG

puuid = MG.my_puuid()
ids = MG.my_match_ids(puuid, 40)

rows = []
for mid in ids:
    path = config.DATA_RAW / f"{mid}.json"
    if not path.exists():
        continue
    row = parse.parse_match(path)
    if row is None:
        continue
    row["matchId"] = mid
    side, champ = MG.my_side_and_champ(mid, puuid)
    row["my_side"], row["my_champ"] = side, F.load_labels()["name"].get(champ, champ)
    rows.append(row)

games = pd.DataFrame(rows)
on_patch = games[games["patch"].isin(config.TARGET_PATCHES)].reset_index(drop=True)
print(f"{len(games)} recent games, {len(on_patch)} on the training patches "
      f"{config.TARGET_PATCHES}")
print(f"\npatch spread of my games: {games['patch'].value_counts().to_dict()}")

Only a minority of my recent games fall on the training patches. I play in bursts, so pulling
*more* games reaches further back into *older* patches rather than adding on-patch ones. This is
a genuine limitation of the personal analysis, not something more data would fix.

In [ ]:
X = F.build_aligned(on_patch, tier, columns, F.load_labels())
blue_prob = model.predict_proba(X)[:, 1]

on_patch["my_win_prob"] = [p if s == 100 else 1 - p for p, s in zip(blue_prob, on_patch["my_side"])]
on_patch["won"] = [(w == 1) == (s == 100) for w, s in zip(on_patch["win"], on_patch["my_side"])]

show = on_patch[["my_champ", "my_side", "my_win_prob", "won"]].copy()
show["my_side"] = show["my_side"].map({100: "blue", 200: "red"})
show["my_win_prob"] = show["my_win_prob"].map("{:.1%}".format)
show

In [ ]:
predicted, actual = on_patch["my_win_prob"].mean(), on_patch["won"].mean()
print(f"games analysed:    {len(on_patch)}")
print(f"draft predicted:   {predicted:.1%}")
print(f"actually won:      {actual:.1%}")
print(f"draft's >50% call was right in "
      f"{(on_patch['won'] == (on_patch['my_win_prob'] > 0.5)).mean():.0%} of games")

In [ ]:
fig, ax = viz.figure(8.4, 3.2)
for won, row in [(True, 1), (False, 0)]:
    sub = on_patch[on_patch["won"] == won]
    ax.plot(sub["my_win_prob"], [row] * len(sub), "o",
            color=viz.WON if won else viz.LOST, ms=11, mec=viz.SURFACE, mew=1.5,
            label=f"{'won' if won else 'lost'} ({len(sub)})")
viz.reference_line(ax, 0.5, "coin flip", axis="x")
ax.set_yticks([0, 1]); ax.set_yticklabels(["LOST", "WON"], fontweight="semibold")
ax.set_ylim(-0.6, 1.6); ax.set_xlim(0.40, 0.60)
ax.set_xticks([.40, .45, .50, .55, .60]); ax.set_xticklabels(["40%", "45%", "50%", "55%", "60%"])
ax.set_xlabel("the draft's read on my chances")
viz.despine_x(ax); ax.legend(loc="lower right", ncol=2)
viz.title_block(ax, "Did a better draft actually mean a win?",
                "If it did, the WON dots would sit right of the LOST dots.")
plt.show()

## What this says

The draft handed me something close to a coin flip in essentially every game, and the wins and
losses land on top of each other rather than separating. The gap between what the draft predicted
and what actually happened is not champion select — it is execution.

**Read the sample size honestly.** This is a handful of games: the exact win rate here is a hot
streak, not a skill estimate (my season win rate is around 53%). The *size of the gap* is the
point, not its precise value.